## Text Classification - Transformer (only self-attention)

In [2]:
import torch
import torch.nn as nn
from tokenizers import Tokenizer
from tokenizers.models import WordLevel
from tokenizers.trainers import WordLevelTrainer
from tokenizers.pre_tokenizers import Whitespace
 
corpus = [
    "gậy ông đập lưng ông",
    "có làm mới có ăn"
]
data_size = len(corpus)

# 0: negative - 1: positive
labels = [0, 1]

# Define the max vocabulary size and sequence length
vocab_size = 8
sequence_length = 5

In [3]:
# Initialize the tokenizer and define a trainer
tokenizer = Tokenizer(WordLevel())
tokenizer.pre_tokenizer = Whitespace()
tokenizer.enable_padding(pad_id=1, 
                         pad_token="<pad>", 
                         length=sequence_length)
tokenizer.enable_truncation(max_length=sequence_length)

# Train the tokenizer on your corpus
trainer = WordLevelTrainer(vocab_size=vocab_size, 
                           special_tokens=["<unk>", "<pad>"])
tokenizer.train_from_iterator(corpus, trainer)

# Tokenize and numericalize your samples
def vectorize(sentence, tokenizer):
    output = tokenizer.encode(sentence)
    return torch.tensor(output.ids, dtype=torch.long)

# Vectorize the samples
corpus_ids = []
for sentence in corpus:
    corpus_ids.append(vectorize(sentence, tokenizer))

In [4]:
for v in corpus_ids:
    print(v)

tensor([4, 3, 0, 6, 3])
tensor([2, 5, 7, 2, 0])


In [5]:
class TextCls_Model(nn.Module):
    def __init__(self, vocab_size, num_classes, embed_dim, num_heads):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim)
        self.attn = nn.MultiheadAttention(embed_dim=embed_dim, 
                                          num_heads=num_heads,
                                          batch_first=True) 
        self.flatten = nn.Flatten()       
        self.classifier = nn.Linear(in_features=10, out_features=2)                             

    def forward(self, x):         # (5)
        x = self.embedding(x)     # (5, 2)
        x,_ = self.attn(x, x, x)  # (5, 2)
        x = self.flatten(x)       # (10,)
        x = self.classifier(x)    # (2,)
        return x

num_classes = 2
model = TextCls_Model(vocab_size, num_classes, 2, 1)

In [6]:
input_1 = torch.tensor([[4, 3, 0, 6, 3]], dtype=torch.long)
label_1 = torch.tensor([0], dtype=torch.long)

output = model(input_1)
print(output.shape)

torch.Size([1, 2])


In [7]:
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.SGD(model.parameters(),
                             lr=0.1)

## Train

In [8]:
inputs = torch.tensor([[4, 3, 0, 6, 3],
                       [2, 5, 7, 2, 0]], dtype=torch.long)
labels = torch.tensor([0, 1], dtype=torch.long)

for _ in range(50):
    optimizer.zero_grad()
    outputs = model(inputs)
    loss = criterion(outputs, labels)
    print(loss)

    loss.backward()
    optimizer.step()

tensor(0.7928, grad_fn=<NllLossBackward0>)
tensor(0.7005, grad_fn=<NllLossBackward0>)
tensor(0.6280, grad_fn=<NllLossBackward0>)
tensor(0.5628, grad_fn=<NllLossBackward0>)
tensor(0.5012, grad_fn=<NllLossBackward0>)
tensor(0.4429, grad_fn=<NllLossBackward0>)
tensor(0.3892, grad_fn=<NllLossBackward0>)
tensor(0.3406, grad_fn=<NllLossBackward0>)
tensor(0.2972, grad_fn=<NllLossBackward0>)
tensor(0.2581, grad_fn=<NllLossBackward0>)
tensor(0.2230, grad_fn=<NllLossBackward0>)
tensor(0.1915, grad_fn=<NllLossBackward0>)
tensor(0.1638, grad_fn=<NllLossBackward0>)
tensor(0.1397, grad_fn=<NllLossBackward0>)
tensor(0.1192, grad_fn=<NllLossBackward0>)
tensor(0.1018, grad_fn=<NllLossBackward0>)
tensor(0.0874, grad_fn=<NllLossBackward0>)
tensor(0.0754, grad_fn=<NllLossBackward0>)
tensor(0.0655, grad_fn=<NllLossBackward0>)
tensor(0.0573, grad_fn=<NllLossBackward0>)
tensor(0.0504, grad_fn=<NllLossBackward0>)
tensor(0.0447, grad_fn=<NllLossBackward0>)
tensor(0.0399, grad_fn=<NllLossBackward0>)
tensor(0.03

## Check if the model has adready learned sucessfuly

In [9]:
outputs = model(inputs)
print(outputs)
print(torch.softmax(outputs, axis=-1))

tensor([[ 3.7866, -1.8074],
        [-2.8877,  1.7110]], grad_fn=<AddmmBackward0>)
tensor([[0.9963, 0.0037],
        [0.0100, 0.9900]], grad_fn=<SoftmaxBackward0>)
